In [1]:
# Notebook: P5_Precheck_v01.ipynb | Cell 1: data checks before Phase 5 (no model changes)
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 80)

ROOT = Path(r"D:\Siam\Thesis_GeoAI")
MASTER = ROOT / "04_master_table" / "master_v06_20261020.csv"
cols = ["Grid_ID", "District", "Upazila_Name", "Is_Water", "Grid_Area_km2", "Elevation_Mean_m", "Population_Count",
        "Building_Count", "Wind_RP100yr_ms", "Storm_Tide_RP100yr_m", "Storm_Tide_Station_Dist_m",
        "Poverty_Rate_Upazila_Pct", "Poverty_SE_Pct", "Distance_to_Hospital_m", "Distance_to_Road_m"]
m = pd.read_csv(MASTER, usecols=cols)
d = m[m["Is_Water"] == 0].copy()
print("model cells:", len(d))
print("NaN per column:", d.isna().sum()[d.isna().sum() > 0].to_dict() or "none")

# 1) Storm tide: values, distance to nearest COAST-RP point, inland copy check
print("\n== Storm tide RP100 ==")
print("unique values:", d["Storm_Tide_RP100yr_m"].nunique())
q = d["Storm_Tide_Station_Dist_m"].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 1.0]) / 1000
print("distance to nearest storm-tide point (km), quantiles:", q.round(1).to_dict())
for km in [10, 20, 50]:
    print(f"  cells > {km} km from a storm-tide point: {(d['Storm_Tide_Station_Dist_m'] > km * 1000).mean() * 100:.1f}%")
exc = np.clip(d["Storm_Tide_RP100yr_m"] - d["Elevation_Mean_m"], 0, None)
print(f"exceedance > 0: {(exc > 0).mean() * 100:.1f}% | median {exc.median():.2f} m")

# 2) District table: population, buildings, wind, tide, access
dist = d.groupby("District").agg(
    Cells=("Grid_ID", "size"), Area_km2=("Grid_Area_km2", "sum"), Pop_Total=("Population_Count", "sum"),
    Buildings=("Building_Count", "sum"), Wind_Mean=("Wind_RP100yr_ms", "mean"), Wind_Min=("Wind_RP100yr_ms", "min"),
    Wind_Max=("Wind_RP100yr_ms", "max"), Tide_Mean=("Storm_Tide_RP100yr_m", "mean"), Tide_Min=("Storm_Tide_RP100yr_m", "min"),
    Tide_Max=("Storm_Tide_RP100yr_m", "max"), Hosp_Max_km=("Distance_to_Hospital_m", lambda s: s.max() / 1000),
    Road_Max_km=("Distance_to_Road_m", lambda s: s.max() / 1000))
print("\n== District table ==\n", dist.round(2))
print(f"TOTAL population (model cells): {d['Population_Count'].sum():,.0f}")

# 3) Poverty per upazila
pov = d.groupby(["District", "Upazila_Name"]).agg(Poverty_Pct=("Poverty_Rate_Upazila_Pct", "first"),
                                                  SE_Pct=("Poverty_SE_Pct", "first"), Cells=("Grid_ID", "size"))
print("\n== Poverty by upazila ==\n", pov.round(2).to_string())
print("upazilas with SE == 0:", int((pov["SE_Pct"] == 0).sum()))

# 4) COAST-RP netCDF metadata (datum check)
nc = list((ROOT / "01_raw").rglob("COAST-RP*.nc"))
print("\n== COAST-RP files ==", [str(p) for p in nc])
if nc:
    import xarray as xr
    ds = xr.open_dataset(nc[0])
    print(ds)
    print("global attrs:", dict(ds.attrs))
    for v in ds.data_vars:
        print(f"  var {v}: attrs {dict(ds[v].attrs)}")
    ds.close()
readmes = [p for p in (ROOT / "01_raw").rglob("*") if p.is_file() and "readme" in p.name.lower()]
print("README files under 01_raw:", [str(p) for p in readmes])

# 5) STORM wind files
sw = [p for p in (ROOT / "01_raw").rglob("*") if p.is_file() and "storm_wind" in str(p).lower()]
print("\n== storm_wind files ==", [p.name for p in sw][:20])

# 6) S1 'Control' evidence
for p in list((ROOT / "03_features").glob("Evid_S1_Control*")) + list((ROOT / "03_features").glob("Label_Flooded_Event_v01*")):
    h = pd.read_csv(p, nrows=5)
    print(f"\n== {p.name} == columns: {list(h.columns)}")
    print(h.head().to_string())
    full = pd.read_csv(p)
    print("rows:", len(full))
    if "Control_Positive" in full.columns:
        print("Control_Positive by event:\n", full.groupby("Event_ID")["Control_Positive"].agg(["count", "sum", "mean"]))
    num = full.select_dtypes("number").columns.drop("Grid_ID", errors="ignore")
    print(full[num].describe().T.round(4).to_string())
print("\nPRECHECK OK")

model cells: 160874
NaN per column: none

== Storm tide RP100 ==
unique values: 15
distance to nearest storm-tide point (km), quantiles: {0.1: 9.8, 0.25: 18.4, 0.5: 36.4, 0.75: 56.2, 0.9: 78.1, 1.0: 101.2}
  cells > 10 km from a storm-tide point: 89.7%
  cells > 20 km from a storm-tide point: 72.6%
  cells > 50 km from a storm-tide point: 32.5%
exceedance > 0: 80.4% | median 1.49 m

== District table ==
             Cells  Area_km2   Pop_Total  Buildings  Wind_Mean  Wind_Min  Wind_Max  Tide_Mean  Tide_Min  Tide_Max  Hosp_Max_km  Road_Max_km
District                                                                                                                                  
Bagerhat    52759   3297.44  1646916.65   299066.0      41.63     38.93     44.26       3.80      2.49      4.53        48.99        27.93
Barguna     20691   1293.19  1063383.63   164957.0      43.44     41.96     45.14       3.76      3.61      3.91        13.85         3.16
Bhola       29471   1841.94  1982597

InvalidVersion: Invalid version: 'unknown'

In [2]:
# %% Cell 2: precheck sections 4-6 only (sections 1-3 already ran). No xarray: broken dask breaks it.
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)

ROOT = Path(r"D:\Siam\Thesis_GeoAI")
RAW = ROOT / "01_raw"
FEAT = ROOT / "03_features"

def clean(v):
    """Make attribute values printable (bytes -> str, small arrays -> list)."""
    if isinstance(v, bytes):
        return v.decode(errors="replace")
    if isinstance(v, np.ndarray):
        return v.tolist() if v.size <= 10 else f"array{v.shape}"
    return v

# ---------- 4) COAST-RP netCDF metadata (vertical datum check) ----------
nc = sorted(RAW.rglob("COAST-RP*.nc"))
print("== COAST-RP files ==", [str(p) for p in nc])
assert nc, "No COAST-RP*.nc found under 01_raw - tell Claude the real file name"
f = nc[0]
with open(f, "rb") as fh:
    magic = fh.read(4)
print("file format magic bytes:", magic)

reader = None
try:
    import netCDF4
    ds = netCDF4.Dataset(f)
    reader = "netCDF4"
    print("dimensions:", {k: len(v) for k, v in ds.dimensions.items()})
    print("global attrs:", {a: clean(ds.getncattr(a)) for a in ds.ncattrs()})
    for name, var in ds.variables.items():
        attrs = {a: clean(var.getncattr(a)) for a in var.ncattrs()}
        rng = ""
        if var.size <= 5e7 and np.issubdtype(var.dtype, np.number):
            arr = np.ma.filled(var[:].astype(float), np.nan)
            rng = f" min={np.nanmin(arr):.3f} max={np.nanmax(arr):.3f}"
        print(f"  var {name} {var.dimensions} {var.shape}{rng}: {attrs}")
    ds.close()
except ImportError:
    print("netCDF4 not installed -> fallback reader")

if reader is None and magic.startswith(b"\x89HDF"):
    import h5py
    reader = "h5py"
    with h5py.File(f, "r") as h:
        print("global attrs:", {k: clean(v) for k, v in h.attrs.items()})
        def visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                attrs = {k: clean(v) for k, v in obj.attrs.items()
                         if k not in ("DIMENSION_LIST", "REFERENCE_LIST")}
                rng = ""
                if obj.size <= 5e7 and np.issubdtype(obj.dtype, np.number):
                    arr = obj[()].astype(float)
                    fv = attrs.get("_FillValue")
                    if fv is not None:
                        arr[arr == np.ravel(fv)[0]] = np.nan
                    rng = f" min={np.nanmin(arr):.3f} max={np.nanmax(arr):.3f}"
                print(f"  var {name} {obj.shape}{rng}: {attrs}")
        h.visititems(visit)

if reader is None and magic.startswith(b"CDF"):
    from scipy.io import netcdf_file
    reader = "scipy"
    ds = netcdf_file(f, "r", mmap=False)
    print("dimensions:", dict(ds.dimensions))
    print("global attrs:", {k: clean(v) for k, v in ds._attributes.items()})
    for name, var in ds.variables.items():
        print(f"  var {name} {var.dimensions} {var.shape}: {({k: clean(v) for k, v in var._attributes.items()})}")
    ds.close()

assert reader is not None, "Could not read the netCDF file with netCDF4, h5py or scipy"
print("reader used:", reader)

readmes = [p for p in RAW.rglob("*") if p.is_file() and "readme" in p.name.lower()]
print("README files under 01_raw:", [str(p) for p in readmes])

# ---------- 5) STORM wind files ----------
sw = [p for p in RAW.rglob("*") if p.is_file() and "storm_wind" in str(p).lower()]
print("\n== storm_wind files ==", [p.name for p in sw][:20])

# ---------- 6) S1 'Control' evidence + raw label file ----------
files = sorted(FEAT.glob("Evid_S1_Control*")) + sorted(FEAT.glob("Label_Flooded_Event_v01*"))
print("\n== S1 files found ==", [p.name for p in files])
for p in files:
    full = pd.read_csv(p)
    print(f"\n== {p.name} == rows: {len(full)} | columns: {list(full.columns)}")
    print(full.head().to_string())
    if "Grid_ID" in full.columns:
        key = ["Grid_ID"] + (["Event_ID"] if "Event_ID" in full.columns else [])
        print("duplicate keys", key, ":", int(full.duplicated(key).sum()))
    if "Control_Positive" in full.columns:
        if "Event_ID" in full.columns:
            print("Control_Positive by event:\n",
                  full.groupby("Event_ID")["Control_Positive"].agg(["count", "sum", "mean"]))
        else:
            print("Control_Positive overall:", full["Control_Positive"].value_counts(dropna=False).to_dict())
    num = full.select_dtypes("number").columns.drop("Grid_ID", errors="ignore")
    print(full[num].describe().T.round(4).to_string())
print("\nPRECHECK SECTIONS 4-6 OK")

== COAST-RP files == ['D:\\Siam\\Thesis_GeoAI\\01_raw\\coast_rp\\COAST-RP.nc']
file format magic bytes: b'\x89HDF'
dimensions: {'stations': 23226}
global attrs: {'title': 'COAST-RP (COastal dAtaset of Storm Tide Return Periods)', 'summary': 'Storm tide is defined as the sum of the storm surge and tidal levels. Tropical cyclone tracks from the STORM dataset and extratropical cyclones from the ERA5 climate reanalysis produced by the ECMWF are used as forcing. Hydrodynamic simulations are carried out with the Global Tide and Surge Model (GTSM).', 'source': 'DFLOWFM / ERA5 climate reanalysis / STORM', 'model_version': 'Deltares, D-Flow FM Version 1.2.63.64757', 'geospatial_lat_min': np.int64(-90), 'geospatial_lat_max': '90', 'geospatial_lat_units': 'degrees_north', 'geospatial_lon_min': '-180', 'geospatial_lon_max': '180', 'geospatial_lon_units': 'degrees_east', 'geospatial_vertical_units': 'm', 'geospatial_vertical_positive': 'up', 'institution': 'IVM - VU Amsterdam', 'creator_email': 'jo

In [4]:
# P5_Risk_SSP_v01.ipynb - copy each "# %%" block into its own Jupyter cell, run in order (Cells 1-7)
# Run ONLY after the precheck output has been reviewed.

# %% Cell 1: settings, inputs, checks
from pathlib import Path
import os, json, time
import numpy as np
import pandas as pd
import geopandas as gpd
import joblib
import lightgbm as lgb
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 60)

# ---------------- SETTINGS ----------------
ROOT = Path(r"D:\Siam\Thesis_GeoAI")
MASTER = ROOT / "04_master_table" / "master_v06_20261020.csv"
GRID = ROOT / "02_boundary_grid" / "grid_250m_v01.gpkg"
V03 = ROOT / "05_models" / "final_v03"
P4DIR = ROOT / "05_models" / "phase4_v01"
OUT = ROOT / "05_models" / "phase5_v01"
FIG = OUT / "figures"
FEATURES = ["Elevation_Mean_m", "Slope_Mean_deg", "Distance_to_River_m", "Cropland_Fraction"]
FEATURES_NOCROP = ["Elevation_Mean_m", "Slope_Mean_deg", "Distance_to_River_m"]
N_MODEL = 160874
SEED = 42
N_MC = 200
N_BOOT = 200
H_W = (1/3, 1/3, 1/3)      # H weights: flood ML, wind, storm-tide exceedance (equal by default)
V_W = 0.5                  # weight of S (poverty) in V; access gets 1 - V_W
N_THREADS = max(1, (os.cpu_count() or 2) // 2)
# IPCC AR6 global mean sea-level rise, m relative to 1995-2014: (median, likely low, likely high)
SLR = {("SSP245", 2050): (0.20, 0.17, 0.26), ("SSP585", 2050): (0.23, 0.20, 0.29),
       ("SSP245", 2100): (0.56, 0.44, 0.76), ("SSP585", 2100): (0.77, 0.63, 1.01)}
# ------------------------------------------
OUT.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)

NEED = ["Grid_ID", "District", "Upazila_Name", "Is_Water", "Grid_Area_km2", "Population_Count", "Building_Count",
        "Elevation_Mean_m", "Slope_Mean_deg", "Distance_to_River_m", "Cropland_Fraction", "Mangrove_Fraction_GMW",
        "Wind_RP100yr_ms", "Storm_Tide_RP100yr_m", "Storm_Tide_Station_Dist_m",
        "Poverty_Rate_Upazila_Pct", "Poverty_SE_Pct", "Distance_to_Hospital_m", "Distance_to_Road_m"]
m = pd.read_csv(MASTER, usecols=NEED)
assert m["Grid_ID"].is_unique
d = m.loc[m["Is_Water"] == 0].copy()
hz = pd.read_csv(V03 / "hazard_predictions_v03.csv",
                 usecols=["Grid_ID", "Hazard_Prob_LGBM", "Is_Within_AOA", "Is_Observable"])
p4 = pd.read_csv(P4DIR / "phase4_shap_ensemble_v01.csv",
                 usecols=["Grid_ID", "Hazard_Rank_Pct_LGBM", "Hazard_Ens_Rank_SD_Pct", "Top10_Frequency_Pct",
                          "Dominant_Increasing_Driver", "Driver_Stability_Pct"])
d = d.merge(hz, on="Grid_ID", validate="1:1").merge(p4, on="Grid_ID", validate="1:1")
assert len(d) == N_MODEL, len(d)

npz = np.load(P4DIR / "ensemble_members_prob_v01.npz")
gid, P = npz["Grid_ID"], npz["P"]
assert len(gid) == N_MODEL and P.shape == (N_MODEL, 50)
d = d.set_index("Grid_ID").loc[gid].reset_index()          # align rows with ensemble matrix
assert (d["Grid_ID"].to_numpy() == gid).all()

core = [c for c in NEED if c not in ("Grid_ID", "District", "Upazila_Name", "Is_Water", "Storm_Tide_Station_Dist_m")]
na = d[core].isna().sum(); na = na[na > 0]
print("NaN in core columns (model cells):", na.to_dict() if len(na) else "none")
assert len(na) == 0, "NaN in core risk inputs -> paste to Claude, do not impute"
print(d[["Population_Count", "Building_Count", "Wind_RP100yr_ms", "Storm_Tide_RP100yr_m", "Storm_Tide_Station_Dist_m",
         "Poverty_Rate_Upazila_Pct", "Poverty_SE_Pct", "Distance_to_Hospital_m", "Distance_to_Road_m", "Grid_Area_km2"]]
      .describe(percentiles=[0.01, 0.5, 0.99]).T[["min", "1%", "50%", "99%", "max"]].round(3))
assert (d["Population_Count"] >= 0).all() and (d["Building_Count"] >= 0).all()
assert d["Wind_RP100yr_ms"].between(0, 120).all(), "wind outside 0-120 m/s"
assert d["Storm_Tide_RP100yr_m"].between(0, 20).all(), "storm tide outside 0-20 m"
assert d["Poverty_Rate_Upazila_Pct"].between(0, 100).all() and (d["Poverty_SE_Pct"] >= 0).all()
assert (d["Distance_to_Hospital_m"] >= 0).all() and (d["Distance_to_Road_m"] >= 0).all()
pv = d.groupby("Upazila_Name")[["Poverty_Rate_Upazila_Pct", "Poverty_SE_Pct"]].nunique()
assert (pv <= 1).all().all(), "poverty not constant within upazila"
print("upazilas:", d["Upazila_Name"].nunique(), "| districts:", d["District"].nunique())

d["Tide_Exceed_m"] = np.clip(d["Storm_Tide_RP100yr_m"] - d["Elevation_Mean_m"], 0, None)
print(f"Storm tide RP100 minus elevation: cells with exceedance > 0: {(d['Tide_Exceed_m'] > 0).mean() * 100:.1f}% | "
      f"median {d['Tide_Exceed_m'].median():.2f} m | max {d['Tide_Exceed_m'].max():.2f} m")
print("DATUM NOTE: FABDEM is EGM2008-referenced; confirm storm-tide datum in the Phase 1 source note.")
print("\nCELL 1 OK")

# %% Cell 2: components and present risk (rank-based, multiplicative)
def ref_pct(x, ref):
    """Mid-rank percentile (0-1) of x within reference distribution ref (calibration-safe, scenario-comparable)."""
    s = np.sort(np.asarray(ref, float)); x = np.asarray(x, float)
    return (np.searchsorted(s, x, "left") + np.searchsorted(s, x, "right")) / (2.0 * len(s))

no_assets = (d["Population_Count"].to_numpy() == 0) & (d["Building_Count"].to_numpy() == 0)
d["H_Flood"] = ref_pct(d["Hazard_Prob_LGBM"], d["Hazard_Prob_LGBM"])
d["H_Wind"] = ref_pct(d["Wind_RP100yr_ms"], d["Wind_RP100yr_ms"])
d["H_Tide"] = ref_pct(d["Tide_Exceed_m"], d["Tide_Exceed_m"])
d["H_Index"] = H_W[0] * d["H_Flood"] + H_W[1] * d["H_Wind"] + H_W[2] * d["H_Tide"]
E = 0.5 * ref_pct(d["Population_Count"], d["Population_Count"]) + 0.5 * ref_pct(d["Building_Count"], d["Building_Count"])
d["E_Index"] = np.where(no_assets, 0.0, E)
d["S_Index"] = ref_pct(d["Poverty_Rate_Upazila_Pct"], d["Poverty_Rate_Upazila_Pct"])
d["AC_Index"] = 1 - 0.5 * (ref_pct(d["Distance_to_Hospital_m"], d["Distance_to_Hospital_m"]) +
                           ref_pct(d["Distance_to_Road_m"], d["Distance_to_Road_m"]))
d["V_Index"] = V_W * d["S_Index"] + (1 - V_W) * (1 - d["AC_Index"])
d["Risk_Index"] = d["H_Index"] * d["E_Index"] * d["V_Index"]
base = d["Risk_Index"].to_numpy()
d["Risk_Rank_Pct"] = ref_pct(base, base) * 100
Q_EDGES = np.quantile(base, [0.2, 0.4, 0.6, 0.8])
TOP_THR = np.quantile(base, 0.9)
d["Risk_Class"] = np.digitize(base, Q_EDGES) + 1
d["Is_Top10_Risk"] = (base > TOP_THR).astype(int)
top0 = d["Is_Top10_Risk"].to_numpy() == 1
pop = d["Population_Count"].to_numpy()
area = d["Grid_Area_km2"].to_numpy()

print("cells with no population and no buildings (E = 0):", int(no_assets.sum()))
print("Spearman among components:\n",
      d[["H_Flood", "H_Wind", "H_Tide", "E_Index", "S_Index", "AC_Index", "V_Index"]].corr(method="spearman").round(2))
print(f"\nTop-10% risk: {top0.sum()} cells, {area[top0].sum():.0f} km2, population {pop[top0].sum() / 1e3:.0f}k "
      f"({pop[top0].sum() / pop.sum() * 100:.1f}% of total) | within AOA {(d.loc[top0, 'Is_Within_AOA'] == 1).mean() * 100:.1f}%")
dist0 = d.groupby("District").agg(Cells=("Grid_ID", "size"), Pop_k=("Population_Count", lambda s: s.sum() / 1e3),
                                  Mean_Risk_Pct=("Risk_Rank_Pct", "mean"), Top10_Cells_Pct=("Is_Top10_Risk", lambda s: s.mean() * 100))
dist0["Top10_Pop_k"] = d[top0].groupby("District")["Population_Count"].sum() / 1e3
print("\n== District, present ==\n", dist0.round(1))
upz0 = (d[top0].groupby(["District", "Upazila_Name"])["Population_Count"].sum() / 1e3).sort_values(ascending=False).head(10)
print("\n== Top 10 upazilas by population in top-10% risk (k) ==\n", upz0.round(1))
print("\nCELL 2 OK")

# %% Cell 3: Monte Carlo (hazard ensemble member + upazila poverty sampling error)
t0 = time.time()
codes, _ = pd.factorize(d["Upazila_Name"])
pov_u = d.groupby(codes)["Poverty_Rate_Upazila_Pct"].first().sort_index().to_numpy()
se_u = d.groupby(codes)["Poverty_SE_Pct"].first().sort_index().to_numpy()
pov_ref = d["Poverty_Rate_Upazila_Pct"].to_numpy()
Hw, Ht = d["H_Wind"].to_numpy(), d["H_Tide"].to_numpy()
Ev, AC = d["E_Index"].to_numpy(), d["AC_Index"].to_numpy()
rng = np.random.default_rng(SEED)
R = np.empty((N_MODEL, N_MC), np.float32)
for k in range(N_MC):
    j = rng.integers(P.shape[1])
    hf = ref_pct(P[:, j], P[:, j])
    pov = np.clip(rng.normal(pov_u, se_u), 0, 100)[codes]
    Vk = V_W * ref_pct(pov, pov_ref) + (1 - V_W) * (1 - AC)
    rk = (H_W[0] * hf + H_W[1] * Hw + H_W[2] * Ht) * Ev * Vk
    R[:, k] = ref_pct(rk, rk) * 100
d["Risk_MC_Rank_Mean_Pct"] = R.mean(1)
d["Risk_MC_Rank_P05_Pct"] = np.percentile(R, 5, axis=1)
d["Risk_MC_Rank_P95_Pct"] = np.percentile(R, 95, axis=1)
d["Risk_Top10_Prob_Pct"] = (R > 90).mean(1) * 100
width = d["Risk_MC_Rank_P95_Pct"] - d["Risk_MC_Rank_P05_Pct"]
print(f"MC done in {time.time() - t0:.0f} s | Spearman(default, MC mean) "
      f"{spearmanr(d['Risk_Rank_Pct'], d['Risk_MC_Rank_Mean_Pct']).correlation:.3f}")
print(f"Median 90% interval width: {width.median():.1f} pct points (top-10% cells: {width[top0].median():.1f})")
print(f"Default top-10% cells with Top10_Prob >= 80%: {(d.loc[top0, 'Risk_Top10_Prob_Pct'] >= 80).mean() * 100:.1f}%")
print("\nCELL 3 OK")

# %% Cell 4: sensitivity analysis incl. cropland-free hazard
def mm(x):
    x = np.asarray(x, float); return (x - x.min()) / (x.max() - x.min())

def risk_from(hf, hw, ht, e, s, ac, wh=H_W, wv=V_W, agg="mult"):
    h = wh[0] * hf + wh[1] * hw + wh[2] * ht
    v = wv * s + (1 - wv) * (1 - ac)
    return h * e * v if agg == "mult" else (h + e + v) / 3.0

def compare(name, alt):
    top_alt = alt > np.quantile(alt, 0.9)
    return {"Scenario": name, "Spearman_vs_Default": round(spearmanr(base, alt).correlation, 3),
            "Top10_Overlap_Pct": round((top0 & top_alt).sum() / top0.sum() * 100, 1),
            "Top10_Pop_k": round(pop[top_alt].sum() / 1e3, 1)}

Hf, S = d["H_Flood"].to_numpy(), d["S_Index"].to_numpy()
rows = [compare("Default (equal H, rank, multiplicative)", base)]
rows.append(compare("H weights flood 0.5 / wind 0.25 / tide 0.25", risk_from(Hf, Hw, Ht, Ev, S, AC, wh=(0.5, 0.25, 0.25))))
rows.append(compare("H = flood ML only", risk_from(Hf, Hw, Ht, Ev, S, AC, wh=(1, 0, 0))))
rows.append(compare("H = flood + wind (no storm tide)", risk_from(Hf, Hw, Ht, Ev, S, AC, wh=(0.5, 0.5, 0.0))))
rows.append(compare("V weights poverty 0.75 / access 0.25", risk_from(Hf, Hw, Ht, Ev, S, AC, wv=0.75)))
rows.append(compare("Aggregation additive (H+E+V)/3", risk_from(Hf, Hw, Ht, Ev, S, AC, agg="add")))
e_mm = np.where(no_assets, 0.0, 0.5 * mm(d["Population_Count"]) + 0.5 * mm(d["Building_Count"]))
ac_mm = 1 - 0.5 * (mm(d["Distance_to_Hospital_m"]) + mm(d["Distance_to_Road_m"]))
rows.append(compare("Normalisation min-max", risk_from(mm(d["Hazard_Prob_LGBM"]), mm(d["Wind_RP100yr_ms"]),
                    mm(d["Tide_Exceed_m"]), e_mm, mm(d["Poverty_Rate_Upazila_Pct"]), ac_mm)))

# cropland-free hazard: same params, same folds as Phase 3
t0 = time.time()
oof = pd.read_csv(V03 / "oof_v03.csv", usecols=["Grid_ID", "Event_ID", "Block_ID", "Fold", "y", "LGBM"])
tr = oof.merge(d[["Grid_ID"] + FEATURES], on="Grid_ID", how="left", validate="m:1")
assert len(tr) == 148719 and tr[FEATURES].notna().all().all()
params = joblib.load(V03 / "final_models_v03.joblib")["LGBM"].get_params()
params.update({"verbose": -1, "n_jobs": N_THREADS})
oof_nc = np.empty(len(tr))
for f in sorted(tr["Fold"].unique()):
    tst = (tr["Fold"] == f).to_numpy()
    mf = lgb.LGBMClassifier(**params).fit(tr.loc[~tst, FEATURES_NOCROP], tr.loc[~tst, "y"])
    oof_nc[tst] = mf.predict_proba(tr.loc[tst, FEATURES_NOCROP])[:, 1]
    print(f"  fold {f} done ({time.time() - t0:.0f} s)", flush=True)
y, p_full = tr["y"].to_numpy(), tr["LGBM"].to_numpy()
auc_full, auc_nc = roc_auc_score(y, p_full), roc_auc_score(y, oof_nc)
assert abs(auc_full - 0.702) < 0.005, f"pooled OOF ROC {auc_full:.3f} does not match Phase 3"
blk = tr["Block_ID"].astype(str).to_numpy(); ub = np.unique(blk)
bidx = {b: np.where(blk == b)[0] for b in ub}
rb = np.random.default_rng(SEED); diffs = []
for _ in range(N_BOOT):
    ii = np.concatenate([bidx[b] for b in rb.choice(ub, len(ub), replace=True)])
    if y[ii].min() == y[ii].max():
        continue
    diffs.append(roc_auc_score(y[ii], p_full[ii]) - roc_auc_score(y[ii], oof_nc[ii]))
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"OOF ROC with cropland {auc_full:.3f} | without {auc_nc:.3f} | diff {auc_full - auc_nc:+.3f} [{lo:+.3f}, {hi:+.3f}]")
m3 = lgb.LGBMClassifier(**params).fit(tr[FEATURES_NOCROP], tr["y"])
d["Hazard_Prob_NoCrop"] = m3.predict_proba(d[FEATURES_NOCROP])[:, 1]
for c in ["Cropland_Fraction", "Mangrove_Fraction_GMW"]:
    print(f"Spearman(hazard, {c}): with cropland {spearmanr(d['Hazard_Prob_LGBM'], d[c]).correlation:+.3f} | "
          f"without {spearmanr(d['Hazard_Prob_NoCrop'], d[c]).correlation:+.3f}")
print(f"Spearman(hazard with, without cropland) = {spearmanr(d['Hazard_Prob_LGBM'], d['Hazard_Prob_NoCrop']).correlation:.3f}")
hf_nc = ref_pct(d["Hazard_Prob_NoCrop"], d["Hazard_Prob_NoCrop"])
rows.append(compare("Cropland-free flood hazard", risk_from(hf_nc, Hw, Ht, Ev, S, AC)))
sens = pd.DataFrame(rows)
print("\n== Sensitivity ==\n", sens.to_string(index=False))
sens.to_csv(OUT / "sensitivity_v01.csv", index=False)
print("\nCELL 4 OK")

# %% Cell 5: SSP sea-level sensitivity (storm-tide exceedance; flood ML, E, V held at present)
exc0 = d["Tide_Exceed_m"].to_numpy()
tide, elev = d["Storm_Tide_RP100yr_m"].to_numpy(), d["Elevation_Mean_m"].to_numpy()
cls0 = d["Risk_Class"].to_numpy()
Vv = d["V_Index"].to_numpy()
rows, dist_rows = [], []
for (ssp, yr), (med, lo_, hi_) in SLR.items():
    for lvl, slr in [("median", med), ("likely_low", lo_), ("likely_high", hi_)]:
        exc = np.clip(tide + slr - elev, 0, None)
        ht_f = ref_pct(exc, exc0)
        risk_f = (H_W[0] * Hf + H_W[1] * Hw + H_W[2] * ht_f) * Ev * Vv
        pct_f = ref_pct(risk_f, base) * 100
        cls_f = np.digitize(risk_f, Q_EDGES) + 1
        top_f = risk_f > TOP_THR
        up = cls_f > cls0
        emerg = top_f & ~top0
        rows.append({"SSP": ssp, "Year": yr, "SLR_Level": lvl, "SLR_m": slr,
                     "Cells_Up_Class_Pct": round(up.mean() * 100, 1), "Area_Up_km2": round(area[up].sum(), 0),
                     "Pop_Up_k": round(pop[up].sum() / 1e3, 0),
                     "Pop_Top10_Present_k": round(pop[top0].sum() / 1e3, 0), "Pop_Top10_Future_k": round(pop[top_f].sum() / 1e3, 0),
                     "Emerging_Hotspot_Cells": int(emerg.sum()), "Emerging_Pop_k": round(pop[emerg].sum() / 1e3, 0),
                     "Mean_Pct_Change": round((pct_f - d["Risk_Rank_Pct"].to_numpy()).mean(), 2)})
        if lvl == "median":
            tag = f"{ssp}_{yr}"
            d[f"Risk_Rank_Pct_{tag}"] = pct_f
            d[f"Risk_Class_{tag}"] = cls_f
            g = pd.DataFrame({"District": d["District"], "chg": pct_f - d["Risk_Rank_Pct"].to_numpy(),
                              "pop": pop, "t0": top0, "tf": top_f})
            for dist_name, gg in g.groupby("District"):
                dist_rows.append({"Scenario": tag, "District": dist_name, "Mean_Pct_Change": round(gg["chg"].mean(), 2),
                                  "Top10_Pop_Present_k": round(gg.loc[gg["t0"], "pop"].sum() / 1e3, 1),
                                  "Top10_Pop_Future_k": round(gg.loc[gg["tf"], "pop"].sum() / 1e3, 1)})
ssp_tab, ssp_dist = pd.DataFrame(rows), pd.DataFrame(dist_rows)
print("== SSP summary ==\n", ssp_tab.to_string(index=False))
print("\n== District change (median SLR) ==\n", ssp_dist.to_string(index=False))
ssp_tab.to_csv(OUT / "ssp_summary_v01.csv", index=False); ssp_dist.to_csv(OUT / "ssp_district_v01.csv", index=False)
print("\nCELL 5 OK")

# %% Cell 6: does the hazard-model residual follow surge/wind forcing?
diag = pd.read_csv(P4DIR / "diag_upazila_v01.csv", usecols=["Upazila_Name", "Obs_Rate", "OOF_Pred", "Obs_to_Pred"])
uf = d.groupby("Upazila_Name").agg(Storm_Tide_m=("Storm_Tide_RP100yr_m", "mean"), Wind_ms=("Wind_RP100yr_ms", "mean"),
                                   Tide_Exceed_m=("Tide_Exceed_m", "mean")).reset_index()
fc = diag.merge(uf, on="Upazila_Name", validate="1:1").dropna(subset=["Obs_to_Pred"])
print(f"n upazilas = {len(fc)}")
for c in ["Storm_Tide_m", "Wind_ms", "Tide_Exceed_m"]:
    r = spearmanr(fc["Obs_to_Pred"], fc[c])
    print(f"  Spearman(Obs_to_Pred, {c:14s}) = {r.correlation:+.3f} (p = {r.pvalue:.3f})")
print(fc.sort_values("Obs_to_Pred", ascending=False).round(3).to_string(index=False))
fc.to_csv(OUT / "residual_vs_forcing_v01.csv", index=False)
print("\nCELL 6 OK")

# %% Cell 7: maps, save outputs, paste-back summary
import matplotlib.pyplot as plt
grid = gpd.read_file(GRID)[["Grid_ID", "geometry"]]
gm = grid.merge(d, on="Grid_ID", how="inner"); assert len(gm) == N_MODEL
gw = grid.merge(m.loc[m["Is_Water"] == 1, ["Grid_ID"]], on="Grid_ID")
gm["SSP585_2100_Change"] = gm["Risk_Rank_Pct_SSP585_2100"] - gm["Risk_Rank_Pct"]

def panel(ax, col, title, cmap, vmin=None, vmax=None):
    gw.plot(ax=ax, color="#d9e6f2", linewidth=0, rasterized=True)
    gm.plot(ax=ax, column=col, cmap=cmap, vmin=vmin, vmax=vmax, linewidth=0, legend=True,
            legend_kwds={"shrink": 0.6}, rasterized=True)
    ax.set_title(title, fontsize=10); ax.set_axis_off()

fig, ax = plt.subplots(1, 3, figsize=(18, 7))
panel(ax[0], "H_Index", "Hazard index H (flood, wind, storm tide)", "YlOrRd", 0, 1)
panel(ax[1], "E_Index", "Exposure index E (population, buildings)", "Purples", 0, 1)
panel(ax[2], "V_Index", "Vulnerability index V (poverty, access)", "Greens", 0, 1)
fig.tight_layout(); fig.savefig(FIG / "Fig_P5_Map_H_E_V.png", dpi=300); plt.close(fig)

fig, ax = plt.subplots(1, 3, figsize=(18, 7))
panel(ax[0], "Risk_Rank_Pct", "Present risk percentile", "YlOrRd", 0, 100)
panel(ax[1], "Risk_Top10_Prob_Pct", "Probability of top-10% risk (Monte Carlo, %)", "magma_r", 0, 100)
panel(ax[2], "SSP585_2100_Change", "Risk percentile change, SSP5-8.5 2100 (median SLR)", "RdBu_r", -20, 20)
fig.tight_layout(); fig.savefig(FIG / "Fig_P5_Map_Risk_MC_SSP.png", dpi=300); plt.close(fig)

save_cols = ["Grid_ID", "Tide_Exceed_m", "H_Flood", "H_Wind", "H_Tide", "H_Index", "E_Index", "S_Index", "AC_Index",
             "V_Index", "Risk_Index", "Risk_Rank_Pct", "Risk_Class", "Is_Top10_Risk", "Risk_MC_Rank_Mean_Pct",
             "Risk_MC_Rank_P05_Pct", "Risk_MC_Rank_P95_Pct", "Risk_Top10_Prob_Pct", "Hazard_Prob_NoCrop"] + \
            [c for c in d.columns if c.startswith("Risk_Rank_Pct_SSP") or c.startswith("Risk_Class_SSP")]
d[save_cols].to_csv(OUT / "phase5_risk_v01.csv", index=False)
(OUT / "phase5_config_v01.json").write_text(json.dumps(
    {"H_weights_flood_wind_tide": H_W, "V_weight_poverty": V_W, "normalisation": "mid-rank percentile",
     "aggregation": "H*E*V", "E_zero_rule": "E=0 where population and buildings are both 0", "n_mc": N_MC,
     "seed": SEED, "slr_source": "IPCC AR6 global mean SLR, rel. 1995-2014 (median, likely range)",
     "slr_m": {f"{k[0]}_{k[1]}": v for k, v in SLR.items()},
     "ssp_note": "flood ML hazard, E and V held at present; only storm-tide exceedance changes"}, indent=1))
print("Saved:", OUT / "phase5_risk_v01.csv", "| figures:", sorted(p.name for p in FIG.glob("*.png")))

print("\n=== PHASE 5 SUMMARY ===")
print(f"Top-10% risk: {top0.sum()} cells, {area[top0].sum():.0f} km2, pop {pop[top0].sum() / 1e3:.0f}k "
      f"({pop[top0].sum() / pop.sum() * 100:.1f}%), within AOA {(d.loc[top0, 'Is_Within_AOA'] == 1).mean() * 100:.1f}%")
print(f"MC: Spearman(default, mean) {spearmanr(d['Risk_Rank_Pct'], d['Risk_MC_Rank_Mean_Pct']).correlation:.3f}; "
      f"top-10% robust (>=80% draws) {(d.loc[top0, 'Risk_Top10_Prob_Pct'] >= 80).mean() * 100:.1f}%")
print("Sensitivity:\n", sens.to_string(index=False))
print("SSP (median):\n", ssp_tab[ssp_tab["SLR_Level"] == "median"].to_string(index=False))
print("\nCELL 7 OK")

NaN in core columns (model cells): none
                              min        1%        50%        99%         max
Population_Count            0.000     0.000     29.089    279.353    1075.015
Building_Count              0.000     0.000      2.000     56.000     293.000
Wind_RP100yr_ms            38.935    39.483     43.149     46.150      46.884
Storm_Tide_RP100yr_m        2.492     2.693      3.746      4.530       4.620
Storm_Tide_Station_Dist_m  85.636  3417.456  36430.866  93962.207  101249.205
Poverty_Rate_Upazila_Pct    7.400     7.400     19.800     47.700      47.700
Poverty_SE_Pct              2.100     2.100      3.400     10.500      14.300
Distance_to_Hospital_m      1.723   499.040   6483.014  41283.815   48989.588
Distance_to_Road_m          0.002     3.557    344.048  24639.491   27925.665
Grid_Area_km2               0.062     0.062      0.062      0.062       0.062
upazilas: 37 | districts: 5
Storm tide RP100 minus elevation: cells with exceedance > 0: 80.4% | media